In [0]:
pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import warnings
warnings.filterwarnings('ignore')

In [0]:
import mlflow

model_config = mlflow.models.ModelConfig(
    development_config="../conf/chapter05_conf.yml"
)

retriever_configs = model_config.get("retriever_configs")

In [0]:
from typing import List, Dict, Optional, Any
from databricks.vector_search.client import VectorSearchClient

from mlflow.entities import SpanType, Document


class VectorSearchWrapper:
    def __init__(self, retriever_config: Dict):
        self.vsc = VectorSearchClient(disable_notice=True)
        self.retriever_cfg = retriever_config

        # Create the index handle
        self.index = self.vsc.get_index(
            endpoint_name=retriever_config["endpoint_name"],
            index_name=retriever_config["index_name"],
        )

    @mlflow.trace(
        span_type=SpanType.RETRIEVER,
        name="single_retriever_search",
        attributes={"vs_type": "databricks_vector_search"},
    )
    def search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ):
        mlflow.update_current_trace(
            tags={
                "vs_endpoint_name": self.retriever_cfg["endpoint_name"],
                "vs_index_name": self.retriever_cfg["index_name"],
            }
        )

        span = mlflow.get_current_active_span()
        span.set_attributes(
            {
                "filters": filters or self.retriever_cfg.get("filters", {}),
                "retriever_k": num_results or self.retriever_cfg.get("k", 5),
            }
        )

        return self.index.similarity_search(
            query_text=query_text,
            columns=columns or self.retriever_cfg.get("columns", []),
            filters=filters or self.retriever_cfg.get("filters", {}),
            num_results=num_results or self.retriever_cfg.get("k", 5),
        )

In [0]:
client = VectorSearchWrapper(retriever_configs['retriever_1'])

# Normal search (docs don't render)
results = client.search("How do I book a flight online with Unity Airways?")

In [0]:
@mlflow.trace(span_type=SpanType.PARSER, name="Parse_Search_Result")
def parse_search_result(self, search_result):
    columns = search_result["manifest"]["columns"]
    data_array = search_result.get("result").get("data_array")

    retriever_schema = self.retriever_cfg["retriever_schema"]

    mapped_result = {}
    output_list = []

    if len(data_array) > 0:
        for data in data_array:
            for column, column_value in zip(columns, data):
                mapped_result[column["name"]] = column_value

            metadata = {"score": mapped_result["score"]}
            doc = Document(
                page_content=mapped_result[retriever_schema["text_column"]],
                metadata=metadata,
                id=mapped_result[retriever_schema["primary_key"]],
            )
            output_list.append(doc)

    return output_list


@mlflow.trace(span_type=SpanType.RETRIEVER)
def refined_search(
    self,
    query_text: str,
    columns: Optional[List[str]] = None,
    filters: Optional[Dict[str, tuple]] = None,
    num_results: Optional[int] = None,
):

    search_results = self.search(
        query_text=query_text,
        columns=columns,
        filters=filters,
        num_results=num_results,
    )

    parsed_results = self.parse_search_result(search_result=search_results)

    return parsed_results


VectorSearchWrapper.parse_search_result = parse_search_result
VectorSearchWrapper.refined_search = refined_search

In [0]:
# Refined search (docs render)
client = client = VectorSearchWrapper(retriever_configs['retriever_1'])
refined_search = client.refined_search("How do I book a flight online with Unity Airways?")

## Multi Retriever

In [0]:
from typing import List, Dict, Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
import contextvars
import mlflow
from mlflow.entities import SpanType
from databricks_langchain import ChatDatabricks
import ast

mlflow.langchain.autolog()


class MultiRetriever:
    def __init__(
        self,
        retriever_configs: List[Dict[str, Any]],
        llm_endpoint: str = "databricks-gpt-oss-120b",
    ):
        self.retrievers = [
            VectorSearchWrapper(retriever_configs[config])
            for config in retriever_configs
        ]
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)

In [0]:
def query_rewriting(self, query_text: str) -> List[str]:
    num_queries = len(self.retrievers)
    response = self.llm.invoke(
        f"Generate {num_queries} variations of this query for vector search: '{query_text}'. "
        f"Return the generated queries in a list. For example ['how are you?', 'how do you do?']"
    )

    text_items = [
        item["text"] for item in response.content if item.get("type") == "text"
    ]

    queries = ast.literal_eval(text_items[0])
    return queries


MultiRetriever.query_rewriting = query_rewriting

In [0]:
@mlflow.trace(span_type=SpanType.RETRIEVER)
def execute_parallel_search(
    self,
    queries: List[str],
    columns: Optional[List[str]] = None,
    filters: Optional[Dict[str, tuple]] = None,
    num_results: Optional[int] = None,
    refined: bool = True,
):
    combined_results: List[Any] = []

    with ThreadPoolExecutor(max_workers=len(self.retrievers)) as executor:
        futures = {}
        for retriever, query in zip(self.retrievers, queries):
            ctx = contextvars.copy_context()
            search_fn = retriever.refined_search if refined else retriever.search
            futures[
                executor.submit(
                    ctx.run, search_fn, query, columns, filters, num_results
                )
            ] = retriever

        for future in as_completed(futures):
            retriever_results = future.result()
            if isinstance(retriever_results, list):
                combined_results.extend(retriever_results)

    return combined_results


MultiRetriever.execute_parallel_search = execute_parallel_search

In [0]:
def query_rewriting_search(
    self, query_text: str, top_k: Optional[int] = None
) -> List[Document]:
    """
    Orchestrate query rewriting, retrieval, and reranking, with optional top_k truncation.
    """
    with mlflow.start_span(name="Custom Retriever", span_type=SpanType.CHAIN) as span:
        span.set_inputs({"query": query_text})

        # Step 1: Generate rewritten queries
        llm_queries = self.query_rewriting(query_text)

        # Step 2: Run parallel retrieval
        search_results = self.execute_parallel_search(llm_queries)

        span.set_outputs(search_results)
        return search_results


MultiRetriever.query_rewriting_search = query_rewriting_search

In [0]:
multi_retriever = MultiRetriever(retriever_configs)
results = multi_retriever.query_rewriting_search(
    query_text="How do I book a flight online with Unity Airways?"
)

## Custom RAG

In [0]:
from mlflow.langchain.langchain_tracer import MlflowLangchainTracer
from langchain_core.messages import BaseMessage
from uuid import UUID


class CustomLangchainTracer(MlflowLangchainTracer):
    def on_chat_model_start(
        self,
        serialized: Dict[str, Any],
        messages: List[List[BaseMessage]],
        *,
        run_id: UUID,
        tags: Optional[List[str]] = None,
        parent_run_id: Optional[UUID] = None,
        metadata: Optional[Dict[str, Any]] = None,
        name: Optional[str] = None,
        **kwargs: Any,
    ):

        if metadata:
            kwargs.update({"reranker_metadata": metadata})

        self._start_span(
            span_name=name or self._assign_span_name(serialized, "reranker model"),
            parent_run_id=parent_run_id,
            span_type=SpanType.RERANKER,
            run_id=run_id,
            inputs=messages,
            attributes=kwargs,
        )

In [0]:
def rerank_results(
    self,
    query_text: str,
    search_results: List[Document],
) -> List[Document]:

    if not search_results:
        return []

    # Extract the text snippets
    docs = [res.page_content for res in search_results]

    # Construct the rerank prompt
    rerank_prompt = (
        f"Given the query:\n'{query_text}'\n\n"
        f"Rerank the following search results in order of relevance. "
        f"Return a JSON list of the ranking of each doc.\n\n"
        f"For example, [2, 3, 1] means: the first doc ranks 2nd, "
        f"the second doc ranks 3rd, and the third doc ranks 1st.\n\n"
        f"Search results:\n{docs}"
    )

    # Invoke the LLM
    response = self.llm.invoke(
        rerank_prompt,
        config={
            "callbacks": [CustomLangchainTracer()],
            "metadata": {"input_docs": len(docs)},
            "run_name": "CustomReranker",
        },
    )

    text_items = [
        item["text"] for item in response.content if item.get("type") == "text"
    ]

    ranking = ast.literal_eval(text_items[0])

    # Build reranked results based on ranking indices
    indexed_results = list(zip(ranking, search_results))
    indexed_results.sort(key=lambda x: x[0])
    reranked_results = [res for _, res in indexed_results]

    return reranked_results


MultiRetriever.rerank_results = rerank_results

In [0]:
@mlflow.trace(span_type="Function")
def apply_top_k_filter(
    self, results: List[Document], top_k: Optional[int] = None
) -> List[Document]:

    if top_k is not None:
        return results[:top_k]
    return results


MultiRetriever.apply_top_k_filter = apply_top_k_filter

In [0]:
def query_rewriting_search(
    self, query_text: str, top_k: Optional[int] = None
) -> List[Document]:
    
    with mlflow.start_span(name="Custom Retriever", span_type=SpanType.CHAIN) as span:
        span.set_inputs({"query": query_text})

        # Step 1: Generate rewritten queries
        llm_queries = self.query_rewriting(query_text)

        # Step 2: Run parallel retrieval
        search_results = self.execute_parallel_search(llm_queries)

        # Step 3: Rerank results
        reranked_results = self.rerank_results(query_text, search_results)

        # Step 4: Apply top_k truncation after rerank
        final_results = self.apply_top_k_filter(reranked_results, top_k)

        span.set_outputs(final_results)
        return final_results
    

MultiRetriever.query_rewriting_search = query_rewriting_search

In [0]:
multi_retriever = MultiRetriever(retriever_configs)
results = multi_retriever.query_rewriting_search(
    query_text="How do I book a flight online with Unity Airways?"
)

## Custom RAG

In [0]:
class CustomRag:
    def __init__(self, retriever, llm_endpoint: str = "databricks-gpt-oss-120b"):
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)
        self.retriever = retriever

    @mlflow.trace(span_type=SpanType.CHAIN, output_reducer=lambda x: "".join(x))
    def llm_call_docs(self, query_text):
        # Extract the text snippets
        docs = self.retriever.query_rewriting_search(query_text)
        docs = [doc.page_content for doc in docs]

        llm_prompt = f"""You are a trusted AI assistant that helps answer questions based only on the provided information. Here is some context which may or may not help you answer the following question: {docs}.
        
        Answer directly, do not repeat the question. Based on this context, answer this question: {query_text}.
        If you don't know the answer, just say that you don't know.
        """
        llm_stream = self.llm.stream(llm_prompt)
        for chunk in llm_stream:
            if isinstance(chunk.content, str):
                yield chunk.content

In [0]:
custom_rag = CustomRag(multi_retriever)

display_str=''
for chunk in custom_rag.llm_call_docs("How do I book a flight online with Unity Airways?"):
    display_str += chunk

display_str

## Low-Level Client APIs

In [0]:
from mlflow import MlflowClient
mlflow_client = MlflowClient()

def low_level_search(
    self,
    query_text: str,
    columns: Optional[List[str]] = None,
    filters: Optional[Dict[str, tuple]] = None,
    num_results: Optional[int] = None,
):
    root_span = None

    # Setup root trace
    root_span = mlflow_client.start_trace(
        name="new_single_retriever_search",
        tags={
            "vs_endpoint_name": self.retriever_cfg["endpoint_name"],
            "vs_index_name": self.retriever_cfg["index_name"],
        },
        inputs={
            "query_text": query_text,
            "columns": columns,
            "filters": filters,
            "num_results": num_results,
        },
        attributes={
            "filters": filters or self.retriever_cfg.get("filters", {}),
            "retriever_k": num_results or self.retriever_cfg.get("k", 5),
        },
        span_type=SpanType.RETRIEVER,
    )

    # Try search
    try:
        search_results = self.index.similarity_search(
            query_text=query_text,
            columns=columns or self.retriever_cfg.get("columns", []),
            filters=filters or self.retriever_cfg.get("filters", {}),
            num_results=num_results or self.retriever_cfg.get("k", 5),
        )

    # Handle Search Error
    except Exception as e:
        mlflow_client.end_trace(
            request_id=root_span.request_id,
            status="ERROR",
            attributes={
                "error_type": type(e).__name__,
                "error_message": str(e),
            },
        )
        raise

    # Search success
    else:
        mlflow_client.end_trace(
            request_id=root_span.request_id,
            outputs=search_results,
            status="OK",
        )
    return search_results

VectorSearchWrapper.low_level_search = low_level_tracing_search

In [0]:
client = VectorSearchWrapper(retriever_configs['retriever_1'])
results = client.new_search("How do I book a flight online with Unity Airways?")